Read the cut solution

In [1]:
from qiskit import qpy
path = 'Experiment_IBM/'
with open(path+'Circuit.qpy', 'rb') as fd:
    qc = qpy.load(fd)[0]

In [3]:
from util import store_solution, read_solution
# store_solution(cut_solution, path+'/Cutqc/cut_solution/')
cut_solution = read_solution(path+'/cut_solution/', qc)
# cut_solution = read_solution(path+'/Cutqc/cut_solution/', qc)

In [4]:
from Cutting_evaluation_Opt import  get_circuit_to_run, pre_recontruction

circuits_to_run, labels ,cutting_downstream_qubits_list, measurement_qubits_list = get_circuit_to_run(cut_solution)

Read the running results of cut subcircuits

In [27]:
import os
import json
import re

# Path to your folder
folder_path = "Experiment_IBM/result_5"

# Get all JSON files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith('.json')]

# Sort files numerically by the subcircuit number
files.sort(key=lambda x: int(re.search(r'\d+', x).group()))

# Read all JSON files in order
pub_result = []
for file in files:
    file_path = os.path.join(folder_path, file)
    with open(file_path, 'r') as f:
        pub_result.append(json.load(f))

# Verify the order
for i, file in enumerate(files):
    print(f"{i}: {file}")


0: IBM_subcircuit0_prob.json
1: IBM_subcircuit1_prob.json
2: IBM_subcircuit2_prob.json
3: IBM_subcircuit3_prob.json
4: IBM_subcircuit4_prob.json
5: IBM_subcircuit5_prob.json
6: IBM_subcircuit6_prob.json
7: IBM_subcircuit7_prob.json
8: IBM_subcircuit8_prob.json
9: IBM_subcircuit9_prob.json
10: IBM_subcircuit10_prob.json
11: IBM_subcircuit11_prob.json
12: IBM_subcircuit12_prob.json
13: IBM_subcircuit13_prob.json
14: IBM_subcircuit14_prob.json
15: IBM_subcircuit15_prob.json
16: IBM_subcircuit16_prob.json
17: IBM_subcircuit17_prob.json
18: IBM_subcircuit18_prob.json
19: IBM_subcircuit19_prob.json
20: IBM_subcircuit20_prob.json
21: IBM_subcircuit21_prob.json
22: IBM_subcircuit22_prob.json
23: IBM_subcircuit23_prob.json
24: IBM_subcircuit24_prob.json


Reconstruct Result Using CutQC

In [28]:
from util import evaluate_circuit
subcircuit_prob = {i:[] for i in range(len(cut_solution['subcircuits']))}
subcircuit_ecr = {i:0 for i in range(len(cut_solution['subcircuits']))}
for i, (c, l) in enumerate(zip(circuits_to_run, labels)):
    subcircuit_prob[l].append(pub_result[i])

In [29]:
from circuit_knitting.cutting.cutqc.wire_cutting_evaluation import  mutate_measurement_basis, measure_prob
from Cutting_evaluation_Opt import modify_subcircuit_instance
from multiprocessing.pool import ThreadPool
from circuit_knitting.cutting.cutqc.wire_cutting import _generate_metadata
from typing import Sequence, Any
from util import reorder_qubits, evaluate_circuit
from qiskit import QuantumCircuit
def _run_subcircuit_batch(
    subcircuit_instance: dict[tuple[tuple[str, ...], tuple[Any, ...]], int],
    subcircuit: QuantumCircuit,
    subcircuit_idx
):
    """
    Execute a circuit using qiskit runtime.

    Args:
        subcircuit_instances: Dictionary containing information about each of the
            subcircuit instances
        subcircuit: The subcircuit to execute
        service: The runtime service
        backend_name: The backends used to execute the subcircuit
        options: Options for the runtime execution of subcircuit

    Returns:
        The measurement probabilities for the subcircuit batch, as calculated from the
        runtime execution
    """
    subcircuit_instance_probs = {}
    circuits_to_run = []
    subcircuit_init_qubits_list=[]
    subcircuit_meas_qubits_list=[]
    subcircuit_meas_basis_list=[]

    # For each circuit associated with a given subcircuit
    for init_meas in subcircuit_instance:
        subcircuit_instance_idx = subcircuit_instance[init_meas]

        # Collect all of the circuits we need to evaluate, ensuring we don't have duplicates
        if subcircuit_instance_idx not in subcircuit_instance_probs:
            modified_subcircuit_instance, init_qubits, meas_qubits = modify_subcircuit_instance(
                subcircuit=subcircuit,
                init=init_meas[0],
                meas=tuple(init_meas[1]),
            )
            circuits_to_run.append(modified_subcircuit_instance)
            subcircuit_init_qubits_list.append(init_qubits)
            subcircuit_meas_qubits_list.append(meas_qubits)
            subcircuit_meas_basis_list.append(tuple(init_meas[1]))
            mutated_meas = mutate_measurement_basis(meas=tuple(init_meas[1]))
            for meas in mutated_meas:
                mutated_subcircuit_instance_idx = subcircuit_instance[
                    (init_meas[0], meas)
                ]
                # Set a placeholder in the probability dict to prevent duplicate circuits to the Sampler
                subcircuit_instance_probs[mutated_subcircuit_instance_idx] = np.array(
                    [0.0]
                )

    # Run all of our circuits in one batch
    # subcircuit_inst_probs = run_subcircuits(
    #     circuits_to_run
    # )
    # subcircuit_inst_probs = run_subcircuits(
    #     circuits_to_run
    # )
    subcircuit_inst_probs = subcircuit_prob[subcircuit_idx]

    # for i, (a,b) in enumerate(zip(subcircuit_inst_probs,subcircuit_inst_probs_2)):
    #     if calculate_fidelity(a,b)<0.988:
    #         print(i, subcircuit_meas_qubits_list[i])

    # Calculate the measured probabilities
    unique_subcircuit_check = {}
    i = 0
    for init_meas in subcircuit_instance:
        subcircuit_instance_idx = subcircuit_instance[init_meas]
        if subcircuit_instance_idx not in unique_subcircuit_check:
            subcircuit_inst_prob = subcircuit_inst_probs[i]
            i = i + 1
            mutated_meas = mutate_measurement_basis(meas=tuple(init_meas[1]))
            for meas in mutated_meas:
                measured_prob = measure_prob(
                    unmeasured_prob=subcircuit_inst_prob, meas=meas
                )
                mutated_subcircuit_instance_idx = subcircuit_instance[
                    (init_meas[0], meas)
                ]
                subcircuit_instance_probs[
                    mutated_subcircuit_instance_idx
                ] = measured_prob
                unique_subcircuit_check[mutated_subcircuit_instance_idx] = True

    return subcircuit_instance_probs

import numpy as np
def run_subcircuit_instances(
    subcircuits: Sequence[QuantumCircuit],
    subcircuit_instances: dict[int, dict[tuple[tuple[str, ...], tuple[Any, ...]], int]],
) -> dict[int, dict[int, np.ndarray]]:
    """
    Execute all provided subcircuits.

    Using the backend(s) provided, this executes all the subcircuits to generate the
    resultant probability vectors.
    subcircuit_instance_probs[subcircuit_idx][subcircuit_instance_idx] = measured probability

    Args:
        subcircuits: The list of subcircuits to execute
        subcircuit_instances: Dictionary containing information about each of the
            subcircuit instances
        service: The runtime service
        backend_names: The backend(s) used to execute the subcircuits
        options: Options for the runtime execution of subcircuits

    Returns:
        The probability vectors from each of the subcircuit instances
    """
    subcircuit_instance_probs: dict[int, dict[int, np.ndarray]] = {}
    with ThreadPool() as pool:
        args = [
            [
                subcircuit_instances[subcircuit_idx],
                subcircuit,
                subcircuit_idx
            ]
            for subcircuit_idx, subcircuit in enumerate(subcircuits)
        ]
        subcircuit_instance_probs_list = pool.starmap(_run_subcircuit_batch, args)

        for i, partition_batch in enumerate(subcircuit_instance_probs_list):
            subcircuit_instance_probs[i] = partition_batch

    return subcircuit_instance_probs

_, _, subcircuit_instances = _generate_metadata(cut_solution)
subcircuit_instance_probs = run_subcircuit_instances(cut_solution['subcircuits'],subcircuit_instances)

In [30]:
from circuit_knitting.cutting.cutqc import (
    reconstruct_full_distribution, reconstruct_dd_full_distribution
)

# CutQC_path ='Result/QAOA/8node_1/CutQC_solution/'
# New_path ='Result/QAOA/8node_1/New_solution/'
# backend_name = 'ibmq_qasm_simulator'

reconstructed_probabilities = reconstruct_full_distribution(
    qc, subcircuit_instance_probs,cut_solution
)

Read the noise uncut of uncut circuit

In [31]:
with open('Experiment_IBM/IBM_qc_prob.json', 'r') as f:
    noise_prob = json.load(f)

Comparing the result

In [ ]:
from util import calculate_fidelity

calculate_fidelity(noise_prob, abs(reconstructed_probabilities))

0.4892018835184846

In [34]:
!pip list

Package               Version
--------------------- ---------
annotated-types       0.6.0
appnope               0.1.4
asttokens             2.4.1
certifi               2024.8.30
cffi                  1.16.0
charset-normalizer    3.3.2
comm                  0.2.2
contourpy             1.2.1
cryptography          42.0.5
cycler                0.12.1
debugpy               1.8.1
decorator             5.1.1
dill                  0.3.8
exceptiongroup        1.2.0
executing             2.0.1
fonttools             4.51.0
ibm-cloud-sdk-core    3.19.2
ibm-platform-services 0.53.2
idna                  3.7
importlib_metadata    7.1.0
ipykernel             6.29.3
ipython               8.22.2
jedi                  0.19.1
jupyter_client        8.6.1
jupyter_core          5.7.2
kiwisolver            1.4.5
matplotlib            3.8.4
matplotlib-inline     0.1.6
mpmath                1.3.0
munkres               1.1.4
nest_asyncio          1.6.0
networkx              3.3
numpy                 1.26.4
pack